# Lesson 4b: Support Vector Machines — Practical Implementation

Building on the theory from Lesson 4a, this notebook demonstrates practical SVM usage with scikit-learn, kernel comparison, and hyperparameter tuning. We will:

1. Load and preprocess the breast cancer dataset
2. Implement SVM with multiple kernel functions
3. Compare kernel performance and visualize decision boundaries
4. Tune hyperparameters (C and γ) using cross-validation
5. Analyze support vectors and margins
6. Evaluate performance with standard metrics

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split, GridSearchCV, cross_val_score, StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, roc_curve, auc
from sklearn.decomposition import PCA

import warnings
warnings.filterwarnings('ignore')

np.random.seed(42)
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 8)

## 1. Data Loading and Exploration

In [ ]:
# Load breast cancer dataset
data = load_breast_cancer()
X = data.data
y = data.target

print(f"Dataset shape: {X.shape}")
print(f"Number of features: {X.shape[1]}")
print(f"Number of samples: {X.shape[0]}")
print(f"Classes: {np.unique(y)}")
print(f"Class distribution: {np.bincount(y)}")
print(f"\nFeature names (first 5): {data.feature_names[:5]}")

In [ ]:
# Data preprocessing: train-test split and standardization
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print(f"Training set size: {X_train_scaled.shape[0]}")
print(f"Test set size: {X_test_scaled.shape[0]}")
print(f"\nFeature scaling statistics:")
print(f"Mean (training): {X_train_scaled.mean(axis=0)[:3]:.6f}")
print(f"Std (training): {X_train_scaled.std(axis=0)[:3]:.6f}")

## 2. SVM with Different Kernels

We implement SVM using different kernel functions: linear, polynomial, RBF (Gaussian), and sigmoid.

### Kernel Functions (from Lesson 4a theory):

**Linear Kernel:** $K(x_i, x_j) = x_i^T x_j$

**Polynomial Kernel:** $K(x_i, x_j) = (\gamma x_i^T x_j + r)^d$

**RBF (Gaussian) Kernel:** $K(x_i, x_j) = \exp(-\gamma \|x_i - x_j\|^2)$ where $\gamma = \frac{1}{2\sigma^2}$

**Sigmoid Kernel:** $K(x_i, x_j) = \tanh(\gamma x_i^T x_j + r)$

The parameter $\gamma$ controls the influence of each training example: large $\gamma$ makes the decision boundary more complex.

In [ ]:
# Train SVM with different kernels
kernels = ['linear', 'poly', 'rbf', 'sigmoid']
kernel_results = {}

for kernel in kernels:
    if kernel == 'poly':
        svm_model = SVC(kernel=kernel, degree=3, C=1.0, gamma='scale', random_state=42)
    else:
        svm_model = SVC(kernel=kernel, C=1.0, gamma='scale', random_state=42)
    
    svm_model.fit(X_train_scaled, y_train)
    y_pred = svm_model.predict(X_test_scaled)
    
    accuracy = accuracy_score(y_test, y_pred)
    precision = precision_score(y_test, y_pred)
    recall = recall_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred)
    
    kernel_results[kernel] = {
        'model': svm_model,
        'accuracy': accuracy,
        'precision': precision,
        'recall': recall,
        'f1': f1,
        'support_vectors': svm_model.n_support_
    }
    
    print(f"\nKernel: {kernel.upper()}")
    print(f"  Accuracy: {accuracy:.4f}")
    print(f"  Precision: {precision:.4f}")
    print(f"  Recall: {recall:.4f}")
    print(f"  F1-Score: {f1:.4f}")
    print(f"  Support vectors: {svm_model.n_support_} ({100*svm_model.n_support_.sum()/len(X_train):.1f}%)")

In [ ]:
# Visualize kernel comparison
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle('SVM Kernel Comparison', fontsize=16, fontweight='bold')

metrics = ['accuracy', 'precision', 'recall', 'f1']
for idx, metric in enumerate(metrics):
    ax = axes[idx // 2, idx % 2]
    values = [kernel_results[k][metric] for k in kernels]
    bars = ax.bar(kernels, values, color=['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728'])
    ax.set_ylabel(metric.capitalize(), fontsize=11)
    ax.set_ylim([0.9, 1.0])
    ax.grid(axis='y', alpha=0.3)
    
    # Add value labels on bars
    for bar in bars:
        height = bar.get_height()
        ax.text(bar.get_x() + bar.get_width()/2., height,
                f'{height:.4f}', ha='center', va='bottom', fontsize=9)

plt.tight_layout()
plt.show()

print("\nKernel comparison shows RBF and polynomial kernels achieve highest accuracy.")

## 3. Hyperparameter Tuning: C and γ

The regularization parameter $C$ controls the trade-off between maximizing the margin and minimizing training error:

- **Small $C$:** Wider margin, more misclassification (underfitting)
- **Large $C$:** Narrow margin, fewer misclassifications (overfitting)

The kernel parameter $\gamma$ controls the reach of each training example in the RBF kernel:

- **Small $\gamma$:** Each support vector has a wide influence (smoother decision boundary)
- **Large $\gamma$:** Each support vector has limited influence (more complex decision boundary)

In [ ]:
# Grid search for optimal C and gamma with RBF kernel
param_grid = {
    'C': [0.1, 1, 10, 100],
    'gamma': [0.001, 0.01, 0.1, 1, 'scale']
}

svm_rbf = SVC(kernel='rbf', random_state=42)
grid_search = GridSearchCV(
    svm_rbf, param_grid,
    cv=StratifiedKFold(n_splits=5, shuffle=True, random_state=42),
    scoring='f1',
    n_jobs=-1,
    verbose=1
)

grid_search.fit(X_train_scaled, y_train)

print(f"\nBest hyperparameters: {grid_search.best_params_}")
print(f"Best cross-validation F1-score: {grid_search.best_score_:.4f}")

In [ ]:
# Analyze grid search results
results_df = pd.DataFrame(grid_search.cv_results_)
results_pivot = results_df.pivot_table(
    values='mean_test_score',
    index='param_C',
    columns='param_gamma'
)

print("\nMean Test F1-Score Grid:")
print(results_pivot.round(4))

# Visualization
fig, ax = plt.subplots(figsize=(10, 6))
sns.heatmap(
    results_pivot, annot=True, fmt='.4f', cmap='RdYlGn',
    cbar_kws={'label': 'F1-Score'}, ax=ax, vmin=0.95, vmax=1.0
)
ax.set_title('Hyperparameter Tuning: C vs γ (F1-Score)', fontsize=14, fontweight='bold')
ax.set_xlabel('Gamma (γ)', fontsize=12)
ax.set_ylabel('Regularization (C)', fontsize=12)
plt.tight_layout()
plt.show()

In [ ]:
# Train final model with best hyperparameters
best_svm = grid_search.best_estimator_
y_pred_best = best_svm.predict(X_test_scaled)
y_pred_proba = best_svm.decision_function(X_test_scaled)

print("\nFinal Model Performance (with tuned hyperparameters):")
print(f"Accuracy: {accuracy_score(y_test, y_pred_best):.4f}")
print(f"Precision: {precision_score(y_test, y_pred_best):.4f}")
print(f"Recall: {recall_score(y_test, y_pred_best):.4f}")
print(f"F1-Score: {f1_score(y_test, y_pred_best):.4f}")
print(f"\nSupport vectors: {best_svm.n_support_.sum()} ({100*best_svm.n_support_.sum()/len(X_train):.1f}%)")

## 4. Decision Boundary Visualization

We visualize the decision boundary in 2D using PCA to reduce the 30-dimensional feature space.

In [ ]:
# Use PCA for 2D visualization
pca = PCA(n_components=2, random_state=42)
X_train_pca = pca.fit_transform(X_train_scaled)
X_test_pca = pca.transform(X_test_scaled)

print(f"Explained variance ratio: {pca.explained_variance_ratio_}")
print(f"Total variance explained: {pca.explained_variance_ratio_.sum():.4f}")

# Train SVM on 2D PCA features for visualization
svm_2d = SVC(
    kernel='rbf',
    C=grid_search.best_params_['C'],
    gamma=grid_search.best_params_['gamma'],
    random_state=42
)
svm_2d.fit(X_train_pca, y_train)

# Create mesh for decision boundary
h = 0.02  # step size in mesh
x_min, x_max = X_train_pca[:, 0].min() - 1, X_train_pca[:, 0].max() + 1
y_min, y_max = X_train_pca[:, 1].min() - 1, X_train_pca[:, 1].max() + 1
xx, yy = np.meshgrid(np.arange(x_min, x_max, h), np.arange(y_min, y_max, h))
Z = svm_2d.decision_function(np.c_[xx.ravel(), yy.ravel()])
Z = Z.reshape(xx.shape)

In [ ]:
# Plot decision boundary with support vectors
fig, ax = plt.subplots(figsize=(12, 8))

# Decision boundary and margins
ax.contourf(xx, yy, Z, levels=20, cmap='RdBu_r', alpha=0.6)
ax.contour(xx, yy, Z, levels=[0], colors='black', linewidths=2)  # Decision boundary
ax.contour(xx, yy, Z, levels=[-1, 1], colors='black', linewidths=1, linestyles='--', alpha=0.5)  # Margins

# Training points
scatter_neg = ax.scatter(
    X_train_pca[y_train == 0, 0], X_train_pca[y_train == 0, 1],
    c='blue', marker='o', s=50, alpha=0.7, label='Class 0', edgecolors='k'
)
scatter_pos = ax.scatter(
    X_train_pca[y_train == 1, 0], X_train_pca[y_train == 1, 1],
    c='red', marker='s', s=50, alpha=0.7, label='Class 1', edgecolors='k'
)

# Support vectors
support_vectors = X_train_pca[svm_2d.support_]
ax.scatter(
    support_vectors[:, 0], support_vectors[:, 1],
    s=200, linewidth=1.5, facecolors='none', edgecolors='green',
    label=f'Support vectors ({len(svm_2d.support_)})', alpha=0.8
)

ax.set_xlabel(f'First Principal Component ({pca.explained_variance_ratio_[0]:.2%} var)', fontsize=11)
ax.set_ylabel(f'Second Principal Component ({pca.explained_variance_ratio_[1]:.2%} var)', fontsize=11)
ax.set_title('SVM Decision Boundary (2D PCA projection)', fontsize=14, fontweight='bold')
ax.legend(loc='best', fontsize=10)
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

print(f"Number of support vectors: {len(svm_2d.support_)} ({100*len(svm_2d.support_)/len(X_train):.1f}%)")

## 5. Effect of Regularization Parameter C

This section demonstrates how the regularization parameter $C$ affects the margin and decision boundary.

In [ ]:
# Train SVMs with different C values
C_values = [0.1, 1, 10, 100]
fig, axes = plt.subplots(2, 2, figsize=(14, 12))
fig.suptitle('Effect of Regularization Parameter C on Decision Boundary', fontsize=14, fontweight='bold')

for idx, C in enumerate(C_values):
    ax = axes[idx // 2, idx % 2]
    
    # Train SVM with current C value
    svm_c = SVC(kernel='rbf', C=C, gamma='scale', random_state=42)
    svm_c.fit(X_train_pca, y_train)
    
    # Decision boundary
    Z_c = svm_c.decision_function(np.c_[xx.ravel(), yy.ravel()]).reshape(xx.shape)
    
    ax.contourf(xx, yy, Z_c, levels=20, cmap='RdBu_r', alpha=0.6)
    ax.contour(xx, yy, Z_c, levels=[0], colors='black', linewidths=2)
    ax.contour(xx, yy, Z_c, levels=[-1, 1], colors='black', linewidths=1, linestyles='--', alpha=0.5)
    
    # Training points
    ax.scatter(X_train_pca[y_train == 0, 0], X_train_pca[y_train == 0, 1],
              c='blue', marker='o', s=30, alpha=0.6, edgecolors='k')
    ax.scatter(X_train_pca[y_train == 1, 0], X_train_pca[y_train == 1, 1],
              c='red', marker='s', s=30, alpha=0.6, edgecolors='k')
    
    # Support vectors
    ax.scatter(X_train_pca[svm_c.support_, 0], X_train_pca[svm_c.support_, 1],
              s=150, linewidth=1.5, facecolors='none', edgecolors='green', alpha=0.8)
    
    accuracy = accuracy_score(y_test, svm_c.predict(X_test_pca))
    ax.set_title(f'C = {C} (SV: {len(svm_c.support_)}, Acc: {accuracy:.4f})', fontsize=11)
    ax.set_xlabel('PC1', fontsize=10)
    ax.set_ylabel('PC2', fontsize=10)
    ax.grid(alpha=0.3)

plt.tight_layout()
plt.show()

## 6. Confusion Matrix and ROC Curve

In [ ]:
# Confusion matrix
cm = confusion_matrix(y_test, y_pred_best)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Confusion matrix heatmap
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=axes[0], 
           xticklabels=['Malignant', 'Benign'],
           yticklabels=['Malignant', 'Benign'])
axes[0].set_ylabel('True Label', fontsize=11)
axes[0].set_xlabel('Predicted Label', fontsize=11)
axes[0].set_title('Confusion Matrix (Test Set)', fontsize=12, fontweight='bold')

# ROC curve
fpr, tpr, thresholds = roc_curve(y_test, y_pred_proba)
roc_auc = auc(fpr, tpr)

axes[1].plot(fpr, tpr, color='#2ca02c', lw=2.5, label=f'ROC curve (AUC = {roc_auc:.4f})')
axes[1].plot([0, 1], [0, 1], 'k--', lw=1, label='Random classifier')
axes[1].set_xlim([0.0, 1.0])
axes[1].set_ylim([0.0, 1.05])
axes[1].set_xlabel('False Positive Rate', fontsize=11)
axes[1].set_ylabel('True Positive Rate', fontsize=11)
axes[1].set_title('ROC Curve (Test Set)', fontsize=12, fontweight='bold')
axes[1].legend(loc='lower right', fontsize=10)
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.show()

print(f"ROC AUC Score: {roc_auc:.4f}")
print(f"\nConfusion Matrix:")
print(f"  True Negatives: {cm[0, 0]}")
print(f"  False Positives: {cm[0, 1]}")
print(f"  False Negatives: {cm[1, 0]}")
print(f"  True Positives: {cm[1, 1]}")

## 7. Comparison: From-Scratch vs Scikit-learn

Compare the scikit-learn SVM performance with the from-scratch implementation from Lesson 4a.

In [ ]:
# Compare kernels with from-scratch implementation understanding
comparison_data = {
    'Implementation': ['Scikit-learn (Linear)', 'Scikit-learn (RBF)', 'Scikit-learn (Poly)'],
    'Accuracy': [
        kernel_results['linear']['accuracy'],
        kernel_results['rbf']['accuracy'],
        kernel_results['poly']['accuracy']
    ],
    'F1-Score': [
        kernel_results['linear']['f1'],
        kernel_results['rbf']['f1'],
        kernel_results['poly']['f1']
    ]
}

comparison_df = pd.DataFrame(comparison_data)
print("\nComparison of Kernel Implementations:")
print(comparison_df.to_string(index=False))

print("\n" + "="*70)
print("Key Insights from Practical Implementation:")
print("="*70)
print("1. RBF kernel generalizes well to this high-dimensional dataset.")
print("2. Regularization parameter C significantly affects model complexity.")
print("3. Feature scaling is essential for SVM (we used StandardScaler).")
print("4. The kernel trick enables efficient computation without explicit mapping.")
print(f"5. Optimal hyperparameters: C={grid_search.best_params_['C']}, γ={grid_search.best_params_['gamma']}")

## 8. Cross-Validation Analysis

Analyze model stability across multiple folds.

In [ ]:
# Cross-validation scores
cv_scores = cross_val_score(
    SVC(kernel='rbf', C=grid_search.best_params_['C'], 
        gamma=grid_search.best_params_['gamma'], random_state=42),
    X_train_scaled, y_train,
    cv=StratifiedKFold(n_splits=5, shuffle=True, random_state=42),
    scoring='f1'
)

print(f"Cross-validation F1 scores: {cv_scores}")
print(f"Mean CV F1-score: {cv_scores.mean():.4f} (+/- {cv_scores.std():.4f})")

# Visualization
fig, ax = plt.subplots(figsize=(10, 6))
folds = np.arange(1, len(cv_scores) + 1)
ax.plot(folds, cv_scores, 'o-', linewidth=2, markersize=8, color='#2ca02c', label='CV F1-Score')
ax.axhline(cv_scores.mean(), color='red', linestyle='--', linewidth=2, label=f'Mean: {cv_scores.mean():.4f}')
ax.fill_between(folds, cv_scores.min(), cv_scores.max(), alpha=0.2, color='green')
ax.set_xlabel('Fold', fontsize=11)
ax.set_ylabel('F1-Score', fontsize=11)
ax.set_title('Cross-Validation Performance Across Folds', fontsize=12, fontweight='bold')
ax.set_xticks(folds)
ax.set_ylim([cv_scores.min() - 0.02, 1.0])
ax.legend(fontsize=10)
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

## 9. Summary and Key Takeaways

### Theory to Practice
1. **Kernel Trick:** The kernel trick efficiently computes dot products in high-dimensional spaces without explicit mapping.
2. **Feature Scaling:** Essential for SVM due to the distance-based nature of the algorithm.
3. **Support Vectors:** Only a subset of training points determine the decision boundary.
4. **Hyperparameter Tuning:** C and γ critically affect model generalization.

### Performance Achieved
- Best accuracy: {:.4f}
- ROC AUC: {:.4f}
- Optimal C: {}
- Optimal γ: {}
- Support vectors: {} ({:.1f}%)

### Practical Lessons
1. RBF kernel provides good generalization for complex, high-dimensional data.
2. GridSearchCV efficiently finds optimal hyperparameters via cross-validation.
3. Decision boundaries visualized in 2D (via PCA) show how C affects margin width.
4. Cross-validation provides robust performance estimates on unseen data.
5. The mathematical concepts from Lesson 4a (Lagrangian duality, kernel trick) enable efficient practical implementation.
""".format(
    accuracy_score(y_test, y_pred_best),
    roc_auc,
    grid_search.best_params_['C'],
    grid_search.best_params_['gamma'],
    len(best_svm.support_),
    100*len(best_svm.support_)/len(X_train)
)